# 1. Environment Setup & Dependency Imports

In this section, we initialize the required environment and import core modules for document extraction, tabular parsing, dense vector retrieval, and local LLM execution.

* **Document Parsing**: `PyMuPDF` (`fitz`) for PDFs and `pandas` for tabular data.
* **Text Chunking**: `RecursiveCharacterTextSplitter` with overlapping windows.
* **Dense Retrieval**: `FAISS` vector store integrated with local `OllamaEmbeddings`.
* **LLM Orchestration**: `ChatOllama` running `llama3.2` locally via LangChain Expression Language (LCEL).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pymupdf 
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import AIMessage, HumanMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings

# 2. Multi-Format Institutional Data Parsing Pipeline

Extracts text and tabular data from institutional reports (Bank Al-Maghrib, HCP, DEPF, Academic Papers).

* **Unstructured Processing (PDF)**: Iterates page-by-page to extract raw text while attaching page-level metadata.
* **Structured Processing (Excel)**: Converts dataframes into Markdown table structures to preserve tabular semantics.

In [ ]:


def parse_all_macro_data(base_dir):
    all_docs = []
    base_path = Path(base_dir)

    for folder in ["BAM", "HCP", "PAPERS"]:
        folder_path = base_path / folder
        if not folder_path.exists():
            continue
            
        for pdf_file in folder_path.glob("*.pdf"):
            try:
                pdf_doc = pymupdf.open(pdf_file)
                for page_num in range(len(pdf_doc)):
                    text = pdf_doc[page_num].get_text("text").strip()
                    if text:
                        all_docs.append(Document(
                            page_content=text,
                            metadata={
                                "file_path": str(pdf_file),
                                "file_name": pdf_file.name,
                                "folder": folder,
                                "page": page_num + 1,
                                "file_type": "pdf"
                            }
                        ))
                pdf_doc.close()
            except Exception as e:
                print(f"Erreur PDF {pdf_file.name}: {e}")

    
    depf_path = base_path / "DEPF"
    if depf_path.exists():
        for excel_file in depf_path.glob("*.xlsx"):
            try:
                sheets = pd.read_excel(excel_file, sheet_name=None)
                for sheet_name, df in sheets.items():
                    df_clean = df.dropna(how='all').dropna(how='all', axis=1)
                    if not df_clean.empty:
                        markdown_table = df_clean.to_markdown(index=False)
                        content = f"# DEPF - {excel_file.name}\n## Feuille: {sheet_name}\n\n{markdown_table}"
                        all_docs.append(Document(
                            page_content=content,
                            metadata={
                                "file_path": str(excel_file),
                                "file_name": excel_file.name,
                                "folder": "DEPF",
                                "sheet_name": sheet_name,
                                "file_type": "excel"
                            }
                        ))
            except Exception as e:
                print(f"Erreur Excel {excel_file.name}: {e}")

    return all_docs

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    PROJECT_ROOT = BASE_DIR.parent
else:
    PROJECT_ROOT = BASE_DIR

DATA_DIR = PROJECT_ROOT / "data"

documents = parse_all_macro_data(DATA_DIR)
print(f"Total de documents/pages extraits : {len(documents)}")

Total de documents/pages extraits : 2687


# 3. PDF Extraction Verification

Sanity check to inspect raw text extractions from target PDFs and ensure character encoding and formatting integrity.

In [ ]:

sample_pdf = PROJECT_ROOT / "data" / "BAM" / "BAM-RA20-Fr.pdf"
if sample_pdf.exists():
    pdf_doc = pymupdf.open(sample_pdf)
    page_text = pdf_doc[24].get_text("text")

    print("--- CONTENU EXTRAIT PAR PYMUPDF ---")
    print(page_text[:1000])
else:
    print(f"Sample PDF not found at path: {sample_pdf}")

# 4. Text Chunking

Splits full-length documents into smaller chunks to maintain precise context windows for embedding generation.

In [ ]:

def chunk_documents(documents):
    splitter=RecursiveCharacterTextSplitter(chunk_size=700,chunk_overlap=150,separators=["\n\n", "\n", ".", " "])
    return splitter.split_documents(documents)
chunked_docs=chunk_documents(documents)         

# 5. Dense Embedding & FAISS Vector Store Indexing
    Text chunks are mapped to high-dimensional dense vector spaces using `nomic-embed-text` via Ollama.
    The generated vectors are stored in a local **FAISS** (Facebook AI Similarity Search) index.

In [ ]:
def create_vector_store(chunked_docs):
    embeddings=OllamaEmbeddings(model="nomic-embed-text")
    vector_store=FAISS.from_documents(chunked_docs,embedding=embeddings)
    return vector_store
vector_store=create_vector_store(chunked_docs)
vectors=vector_store.index.reconstruct_n(0,5)
print(np.array(vectors).shape)
print(vectors[0][:10])
    

# 6. Similarity Search with Threshold Filtering
Retrieves top-k documents and applies a minimum relevancy score cut-off to prevent hallucinations

In [ ]:

def generate_answer(vector_store,query,k=5 ,score_threshold=0.3):
   docs=vector_store.similarity_serach_with_relevance_scores(query,k=k)
   filtred_docs=[doc for doc,score in docs if score>= score_threshold]
   if not filtred_docs:
      return "No useful content"
   context_blocks=[]
   for doc in filtred_docs:
      source=doc.metadata.get('source','unknown')
      page=doc.metadata.get('page',"N/A")
      context_blocks.append(f"[Source:{source} | Page:{page}]\n{doc.page_content}")
   return  "\n\n---\n\n".join(context_blocks) 
    


# 7. Stateless RAG Pipeline Execution
Construction of a single-turn LCEL (LangChain Expression Language) runnable chain.
 **Strict Grounding Prompt**: Instructs the LLM (`llama3.2`) to answer strictly from context and explicitly declare missing information.
Uses token streaming for real-time output rendering.

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity", search_kwargs={"k": 5}
)
SYSTEM_PROMPT = """ Answer the user's question based ONLY on the provided context below.
If the answer cannot be found in the context, state clearly: "I cannot find this information in the provided document."
Do not attempt to invent, extrapolate, or use outside knowledge.
Context : {context}
Question : {question}
Answer : """
prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT)
llm = ChatOllama(model="llama3.2", temperature=0)
output_parser=StrOutputParser() 
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Source: {d.metadata.get('source', 'N/A')} | Page: {d.metadata.get('page', 'N/A')}]\n{d.page_content}"
        for d in docs
    )
rag_chain = ( {"context":retriever|format_docs, "question":RunnablePassthrough()} | prompt | llm | output_parser)
for chunk in rag_chain.stream("Quel est le résultat net de Bank Al-Maghrib en 2020 ?")
print(chunk , end ="",flush=True)




# 8. Conversational RAG with Contextual Query Rewriting
    To handle multi-turn dialogue with coreference resolution (e.g., *'Et quel était son montant en 2019 ?'*):
1. **Query Rewriter**: Reformulates follow-up queries into standalone search inputs using past conversation history.
2. **Contextual Generation**: Executes retrieval with the standalone query and answers the original prompt in conversation context.

In [ ]:

REWRITE_SYSTEM_PROMPT = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""

rewrite_prompt= ChatPromptTemplate.from_messages([("system",REWRITE_SYSTEM_PROMPT),MessagesPlaceholder("chat_history"),("human","{input}")])


QA_SYSTEM_PROMPT = """Answer the user's question based ONLY on the provided context below.
If the answer cannot be found in the context, state clearly: "I cannot find this information in the provided document."
Do not attempt to invent, extrapolate, or use outside knowledge.
Context : \n{context}"""

qa_prompt=ChatPromptTemplate.from_messages([("system",QA_SYSTEM_PROMPT),MessagesPlaceholder("chat_history"),("human","{input}")])

llm = ChatOllama(model="llama3.2", temperature=0)
output_parser=StrOutputParser()
question_rewriter = rewrite_prompt | llm | output_parser
def format_docs(docs):
    if not docs : 
        return "No context found"
    return  "\n\n---\n\n".join(
        f"[Source: {d.metadata.get('source', 'Unknown')} | Page: {d.metadata.get('page', 'N/A')}]\n{d.page_content}"
        for d in docs
    )   
def conversational_rag(user_input, chat_history,retriever):
    if chat_history : 
        standalone_question=question_rewriter.invoke({"chat_history": chat_history,"input":user_input})
    else : 
        standalone_question=user_input

    docs=retriever.invoke(standalone_question)
    context = format_docs(docs)
    
    rag_chain= qa_prompt|llm|output_parser
    final_response = "" 
    for chunk in rag_chain.stream({"context":context,"chat_history":chat_history,"input":user_input}):
        print(chunk,end="",flush=True)
        final_response+= chunk
    
    print()    
    return final_response    
             
        


# 9. Main Pipeline Execution & Multi-Turn Verification 
    Entry point verifying end-to-end multi-turn conversation over local data directory structure.

In [ ]:
if __name__ == "__main__":
 
  DATA_DIR = os.getenv("DATA_DIR", "./data")

  if not os.path.exists(DATA_DIR):
    print(
        f"Data directory '{DATA_DIR}' not found. Please create it and add source files."
    )
  else:
    print("--- Parsing Data Documents ---")
    raw_documents = parse_all_macro_data(DATA_DIR)
    print(f"Total extracted documents/pages: {len(raw_documents)}")

    print("\n--- Chunking Documents ---")
    chunks = chunk_documents(raw_documents)
    print(f"Total chunks created: {len(chunks)}")

    print("\n---Generating Embeddings & FAISS Vector Store ---")
    vector_store = create_vector_store(chunks)
    retriever = vector_store.as_retriever(
        search_type="similarity", search_kwargs={"k": 5}
    )

    print("\n---Testing Conversational RAG Pipeline ---")
    chat_history = []

    
    query_1 = "Quel est le résultat net de Bank Al-Maghrib en 2020 ?"
    print(f"\nUser: {query_1}")
    print("Assistant: ", end="")

    response_1 = conversational_rag(query_1, chat_history, retriever)
    chat_history.append(HumanMessage(content=query_1))
    chat_history.append(AIMessage(content=response_1))

    
    query_2 = "Et quel était son montant en 2019 ?"
    print(f"\nUser: {query_2}")
    print("Assistant: ", end="")

    response_2 = conversational_rag(query_2, chat_history, retriever)